# 4 · A family tree: self-relations, cycles and reachability

**The problem.** A genealogy app. People have parents and children, some people mentor others, and some children
have a legal guardian that may be a person *or* an institution. In a typical object model, "parents" and "children"
are two lists that must be kept in sync by hand. Cycles (a mentor mentoring their own parent) break naive
serializers. And "load everyone related to Ada" is a query you'd write yourself.

With relations:

- a relation can link two objects of the **same** schema,
- "parents" and "children" are two views of **one** relation, so they can't disagree,
- cycles and self-loops are ordinary data, and
- `Reachable.of(x)` answers "everything connected to x".

In [1]:
from mbse.Schemas.Framework import Proxies, Reachable, Schemas
from toolkit import entries

store = Proxies.OfStore()

## One relation, two adjacencies on the same schema

`Parentage` links a `parent` and a `child`. `Person` declares *both* sides, under different accessor names. Links
are untyped, which is what makes this possible: nothing says a parent and a child must be different kinds of thing.

In [2]:
Person = (
    Schemas.OfObject.Builder().ref()
    .properties(
        lambda prop: prop.name('name').of(lambda t: t.as_native(str)),
        lambda prop: prop.name('born').of(lambda t: t.as_native(int)),  # year
    )
    .create()
)
Parentage = (
    Schemas.OfRelation.Builder()
    .links('parent', 'child')
    .properties(lambda prop: prop.name('kind').of(lambda t: t.as_native(str)))  # 'biological', 'adoptive'
    .create()
)
Mentorship = Schemas.OfRelation.Builder().links('mentor', 'mentee').create()

Person = (
    Schemas.OfObject.Builder(Person)
    .relations(
        lambda adj: adj.name('children').of(Parentage).me('parent'),
        lambda adj: adj.name('parents').of(Parentage).me('child'),
        lambda adj: adj.name('mentees').of(Mentorship).me('mentor'),
        lambda adj: adj.name('mentors').of(Mentorship).me('mentee'),
    )
    .update()
)
for name, schema in [('Person', Person), ('Parentage', Parentage), ('Mentorship', Mentorship)]:
    assert schema.validate() == []
    store.register(name, schema)


def names(person, adjacency, link):
    return sorted(e[link].name for e in entries(person, adjacency))

## Building a family from either side

Adding a child from the parent's side and adding a parent from the child's side are the same operation. The
accessor you use only decides which link the object you're building fills:

In [3]:
ada = store.Person().name('Ada').born(1815).create()

byron = (
    store.Person()
    .name('Byron')
    .born(1836)
    .parents(lambda x: x.parent(ada).kind('biological'))  # added from the child's side
    .create()
)
store.Person(ada).children(
    lambda x: x.child(lambda c: c.name('Annabella').born(1837)).kind('biological')  # from the parent's side, inline
).update()

print('Ada\'s children:   ', names(ada, 'children', 'child'))
print('Byron\'s parents:  ', names(byron, 'parents', 'parent'))

Ada's children:    ['Annabella', 'Byron']
Byron's parents:   ['Ada']


There's no "sync the inverse list" step to forget: `children` and `parents` are two windows onto the same entries.

## Several relations between the same pair, and more than one owner

A person can have two parents (two entries), and the same two people can be related in two different ways (two
relations). None of these interfere:

In [4]:
william = store.Person().name('William').born(1805).create()
store.Person(byron).parents(lambda x: x.parent(william).kind('biological')).update()
store.Person(ada).mentees(lambda x: x.mentee(byron)).update()

print('Byron\'s parents:', names(byron, 'parents', 'parent'))
print('Byron\'s mentors:', names(byron, 'mentors', 'mentor'))

Byron's parents: ['Ada', 'William']
Byron's mentors: ['Ada']


## Cycles and self-loops are just data

Byron mentors his mother (a cycle through two relations), and William mentors himself (a self-loop):

In [5]:
store.Person(byron).mentees(lambda x: x.mentee(ada)).update()
store.Person(william).mentees(lambda x: x.mentee(william)).update()

print('William mentors:', names(william, 'mentees', 'mentee'), '| mentored by:', names(william, 'mentors', 'mentor'))

William mentors: ['William'] | mentored by: ['William']


## Everything connected: `Reachable.of`

`Reachable.of(root)` returns the root and every object reachable from it through *any* adjacency, in the order they
were first reached (breadth-first). Cycles are visited once. It follows links in both directions, because an entry
is visible from every end. So it finds the whole *connected component*, not just "descendants":

In [6]:
[p.name for p in Reachable.of(byron)]

['Byron', 'Ada', 'William', 'Annabella']

In [7]:
loner = store.Person().name('Eve').create()
[p.name for p in Reachable.of(loner)]

['Eve']

**Why does this matter?** It's exactly the set of objects you must save together so that every link in them still
points at something. Case study 6 uses it to save and load whole graphs. It doesn't recurse, so a 100,000-person
chain won't hit Python's recursion limit.

## When a link can be filled by different kinds of thing

Guardianship: a child's guardian may be a `Person` or an `Institution`. Both declare an adjacency filling `guardian`.
That's perfectly legal: links are untyped, and a link can be filled by any object whose schema says it can fill it.

In [8]:
Institution = Schemas.OfObject.Builder().ref().properties(lambda prop: prop.name('name').of(lambda t: t.as_native(str))).create()
Guardianship = Schemas.OfRelation.Builder().links('guardian', 'ward').create()

Institution = Schemas.OfObject.Builder(Institution).relations(lambda adj: adj.name('wards').of(Guardianship).me('guardian')).update()
Person = (
    Schemas.OfObject.Builder(Person)
    .relations(
        lambda adj: adj.name('wards').of(Guardianship).me('guardian'),
        lambda adj: adj.name('guardians').of(Guardianship).me('ward'),
    )
    .update()
)
store.register('Institution', Institution)
store.register('Guardianship', Guardianship)

Linking existing objects works for either kind:

In [9]:
court = store.Institution().name('Court of Chancery').create()
store.Person(byron).guardians(lambda x: x.guardian(court)).update()
store.Person(byron).guardians(lambda x: x.guardian(william)).update()
sorted((e['guardian'].schema_name(), e['guardian'].name) for e in entries(byron, 'guardians'))

[('Institution', 'Court of Chancery'), ('Person', 'William')]

But building the guardian *inline* is refused. With two schemas able to fill `guardian`, the framework would have
to guess which one you meant, and it won't:

In [10]:
try:
    store.Person(byron).guardians(lambda x: x.guardian(lambda g: g.name('?'))).update()
except TypeError as error:
    print('TypeError:', error)

TypeError: expected exactly one object schema filling link 'guardian', found ['Person', 'Institution']


Build it first with the builder you mean (`store.Institution()...create()`), then link it. Refusing to guess is
a recurring choice in this framework. The alternative, silently picking the first match, would work until someone
registered a new schema, and then change meaning.

`schema_name()` tells you which kind an object is. Every instance carries it, and case study 6 shows it travelling
with references on the wire.

## What you learned

- A relation can link objects of one schema. Each side is an adjacency with its own accessor name.
- Inverse views (`parents`/`children`) are the same entries, so they're always consistent.
- Multiple owners, several relations per pair, cycles and self-loops need no special handling.
- `Reachable.of(x)` lists x's connected component, root first, in first-reference order.
- A link filled by several schemas accepts any of them. Inline creation needs exactly one.

**Next:** case study 5 adds rules. It covers *how many* entries a relation may have (a price per currency, one
owner per pet) and how to check data against its schema when you decide it's time.